# DBRepo Data Loading & View Verification

This notebook:
1. Loads and cleans the raw CSV data
2. Transforms it to match the 3NF schema
3. Inserts all rows into DBRepo via REST API
4. Verifies all seven SQL views defined in `views.sql`

**3NF Schema:**
```
district         (district_id PK, nuts_code VARCHAR(5), district_code INT UNIQUE)
measurement_info (district_id FK, reference_date DATE, measurement_id UNIQUE, population_avg INT)
unemployment     (measurement_id FK, gender ENUM, value INT, density DECIMAL)
tourism          (measurement_id FK, value INT, density DECIMAL)
```

**Important** <br>
The notebook `02_load_views_dbrepo.ipynb` first has to be executed once to make sure that the SQL views that are used here for verification exist on DBRepo.

## 0  · Configuration

In [1]:
import os, time, math
import pandas as pd
import numpy as np
import requests
from dotenv import load_dotenv

load_dotenv()

ENDPOINT = "https://test.dbrepo.tuwien.ac.at"
DB_ID = "412fb0ce-5299-4d0e-a271-4641b1365b8a"
TOURISM_TABLE_ID = "fa5fe819-9b20-422a-a6fe-299ba0043d87"
UNEMPLOYMENT_TABLE_ID = "5a3f74b9-8a62-4f33-9486-6c6cd3d34cd2"
MEASUREMENT_INFO_TABLE_ID = "5d6d28a7-f211-4bc4-9ada-94450cd0fe6f"
DISTRICT_TABLE_ID = "886fba43-0570-4dda-8233-4f73473ca0a4"
USERNAME = os.getenv("DBREPO_USERNAME")
PASSWORD = os.getenv("DBREPO_PASSWORD")

AUTH = (USERNAME, PASSWORD)
BASE_URL = f"{ENDPOINT}/api/v1/database/{DB_ID}"
HEADERS = {"Content-Type": "application/json", "Accept": "application/json"}

# Path to raw CSV files
TOURISM_CSV = "../data/vienna_tourism_since_2002_raw_v1.0.csv"
UNEMPLOYMENT_CSV = "../data/vienna_unemployment_since_2002_raw_v1.0.csv"

print(f"Target DB : {DB_ID}")
print(f"Endpoint  : {ENDPOINT}")
print(f"User      : {USERNAME}")

Target DB : 412fb0ce-5299-4d0e-a271-4641b1365b8a
Endpoint  : https://test.dbrepo.tuwien.ac.at
User      : Binw3g


## 1  · Helper utilities

In [2]:
def insert_rows(table_id: str, rows: list[dict]) -> None:
    """Insert rows one-by-one into a DBRepo table."""
    url = f"{BASE_URL}/table/{table_id}/data"
    total = len(rows)
    print(f"  Inserting {total} rows into '{table_id}'...")

    for idx, row in enumerate(rows):
        res = requests.post(url, headers=HEADERS, auth=AUTH, json={"data": row})

        if not res.ok:
            raise RuntimeError(
                f"[{table_id}] row {idx+1}/{total} failed "
                f"({res.status_code}): {res.text[:300]}"
            )
        if (idx + 1) % 50 == 0 or (idx + 1) == total:
            print(f"    {idx+1}/{total} rows inserted", end="\r")

        time.sleep(0.05)
    print(f"  ✓ '{table_id}' done.")

## 2  · Load & clean raw CSVs

In [3]:
# tourism
tou_raw = pd.read_csv(
    TOURISM_CSV,
    sep=";",
    skiprows=1,
    decimal=",",
    thousands="."
).dropna(axis=1, how="all")

tou_raw.columns = tou_raw.columns.str.strip()

# REF_DATE is YYYYMMDD (int) -> convert to ISO date string
tou_raw["REF_DATE"] = pd.to_datetime(tou_raw["REF_DATE"].astype(str), format="%Y%m%d").dt.date
tou_raw["NUTS"] = tou_raw["NUTS"].str.strip()

print(f"Tourism rows loaded : {len(tou_raw)}")
print(f"Years               : {sorted(tou_raw['REF_YEAR'].unique())}")
tou_raw.head(3)

Tourism rows loaded : 552
Years               : [np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


,NUTS,DISTRICT_CODE,SUB_DISTRICT_CODE,REF_YEAR,REF_DATE,TOU_VALUE,TOU_DENSITY,POP_AVE
0,AT13,90000,90000,2002,2002-12-31,7655391,4836.48,1582844
1,AT13,90100,90100,2002,2002-12-31,1525278,86662.91,17600
2,AT13,90200,90200,2002,2002-12-31,654979,7317.55,89508


In [4]:
# unemployment
SEX_MAP = {0: "Both", 1: "Male", 2: "Female"}

uep_raw = pd.read_csv(
    UNEMPLOYMENT_CSV,
    sep=";",
    skiprows=1,
    decimal=",",
    thousands="."
).dropna(axis=1, how="all")

uep_raw.columns = uep_raw.columns.str.strip()
uep_raw["GENDER"] = uep_raw["SEX"].map(SEX_MAP)

# REF_DATE is just a year integer; build Dec 31 date to align with tourism convention
uep_raw["REF_DATE"] = pd.to_datetime(uep_raw["REF_DATE"].astype(str) + "1231", format="%Y%m%d").dt.date
uep_raw["NUTS"] = uep_raw["NUTS"].str.strip()

print(f"Unemployment rows loaded : {len(uep_raw)}")
print(f"Years                    : {sorted(uep_raw['REF_YEAR'].unique())}")
print(f"Gender values            : {uep_raw['GENDER'].unique()}")
uep_raw.head(3)

Unemployment rows loaded : 1584
Years                    : [np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Gender values            : ['Both' 'Male' 'Female']


,NUTS,DISTRICT_CODE,SUB_DISTRICT_CODE,REF_YEAR,REF_DATE,SEX,UEP_VALUE,UEP_DENSITY,GENDER
0,AT13,90000,90000,2002,2002-12-31,0,74894,67.83,Both
1,AT13,90100,90100,2002,2002-12-31,0,433,35.32,Both
2,AT13,90200,90200,2002,2002-12-31,0,4784,76.98,Both


## 3  · Build normalized tables

In [5]:
# district
# One row per unique DISTRICT_CODE; district_id is the position in the sorted list.
district_codes = sorted(
    set(tou_raw["DISTRICT_CODE"]).union(uep_raw["DISTRICT_CODE"])
)

# Get NUTS code per district from tourism
nuts_lookup = (
    tou_raw[["DISTRICT_CODE", "NUTS"]]
    .drop_duplicates(subset="DISTRICT_CODE")
    .set_index("DISTRICT_CODE")["NUTS"]
    .to_dict()
)

district_df = pd.DataFrame({
    "district_id": range(1, len(district_codes) + 1),
    "nuts_code": [nuts_lookup[c] for c in district_codes],
    "district_code": district_codes,
})
district_id_map = dict(zip(district_df["district_code"], district_df["district_id"]))

print(f"district rows : {len(district_df)}")
district_df.head()

district rows : 24


,district_id,nuts_code,district_code
0,1,AT13,90000
1,2,AT13,90100
2,3,AT13,90200
3,4,AT13,90300
4,5,AT13,90400


In [6]:
# measurement_info
# One row per (district_code, reference_date) pair
mi_base = (
    tou_raw[["DISTRICT_CODE", "REF_DATE", "POP_AVE"]]
    .drop_duplicates(subset=["DISTRICT_CODE", "REF_DATE"])
    .sort_values(["DISTRICT_CODE", "REF_DATE"])
    .reset_index(drop=True)
)
mi_base["measurement_id"] = range(1, len(mi_base) + 1)
mi_base["district_id"] = mi_base["DISTRICT_CODE"].map(district_id_map)

measurement_df = mi_base[["district_id", "REF_DATE", "measurement_id", "POP_AVE"]].rename(
    columns={"REF_DATE": "reference_date", "POP_AVE": "population_avg"}
)

# Build lookup: (district_code, ref_date) -> measurement_id
meas_id_map = {
    (row.DISTRICT_CODE, row.REF_DATE): row.measurement_id
    for row in mi_base.itertuples()
}

print(f"measurement_info rows : {len(measurement_df)}")
measurement_df.head()

measurement_info rows : 552


,district_id,reference_date,measurement_id,population_avg
0,1,2002-12-31,1,1582844
1,1,2003-12-31,2,1600884
2,1,2004-12-31,3,1620266
3,1,2005-12-31,4,1641646
4,1,2006-12-31,5,1656615


In [7]:
# tourism
tou_df = tou_raw.copy()
tou_df["measurement_id"] = tou_df.apply(
    lambda r: meas_id_map.get((r["DISTRICT_CODE"], r["REF_DATE"])), axis=1
)
tourism_df = tou_df[["measurement_id", "TOU_VALUE", "TOU_DENSITY"]].rename(
    columns={"TOU_VALUE": "value", "TOU_DENSITY": "density"}
)

print(f"tourism rows : {len(tourism_df)}")
tourism_df.head()

tourism rows : 552


,measurement_id,value,density
0,1,7655391,4836.48
1,24,1525278,86662.91
2,47,654979,7317.55
3,70,781717,9498.53
4,93,315284,10789.96


In [8]:
# unemployment
uep_df = uep_raw.copy()
uep_df["measurement_id"] = uep_df.apply(
    lambda r: meas_id_map.get((r["DISTRICT_CODE"], r["REF_DATE"])), axis=1
)
unemployment_df = uep_df[["measurement_id", "GENDER", "UEP_VALUE", "UEP_DENSITY"]].rename(
    columns={"GENDER": "gender", "UEP_VALUE": "value", "UEP_DENSITY": "density"}
)

print(f"unemployment rows : {len(unemployment_df)}")
unemployment_df.head()

unemployment rows : 1584


,measurement_id,gender,value,density
0,1,Both,74894,67.83
1,24,Both,433,35.32
2,47,Both,4784,76.98
3,70,Both,3957,68.25
4,93,Both,1140,55.68


## 4  · Sanity checks before insert

In [9]:
errors = []

# No NULLs in PKs
for df, col, name in [
    (district_df,     "district_id",   "district"),
    (measurement_df,  "measurement_id", "measurement_info"),
    (tourism_df,      "measurement_id", "tourism"),
    (unemployment_df, "measurement_id", "unemployment"),
]:
    n_null = df[col].isna().sum()
    if n_null:
        errors.append(f"{name}.{col}: {n_null} NULL values")

# Referential integrity: all tourism/unemployment measurement_ids exist
valid_mids = set(measurement_df["measurement_id"])
for df, name in [(tourism_df, "tourism"), (unemployment_df, "unemployment")]:
    orphans = set(df["measurement_id"].dropna()) - valid_mids
    if orphans:
        errors.append(f"{name}: {len(orphans)} orphaned measurement_ids")

# Gender enum check
allowed_genders = {"Male", "Female", "Both"}
bad_genders = set(unemployment_df["gender"].unique()) - allowed_genders
if bad_genders:
    errors.append(f"unemployment.gender: unexpected values {bad_genders}")

if errors:
    for e in errors:
        print(f"  ✗ {e}")
    raise ValueError("Pre-insert checks failed")
else:
    print("✓ All pre-insert sanity checks passed.")

✓ All pre-insert sanity checks passed.


## 5  · Delete existing data

Before re-inserting, all existing rows must be deleted in reverse FK order: `unemployment` -> `tourism` -> `measurement_info` -> `district`.

The API endpoint `DELETE /api/v1/database/{databaseId}/table/{tableId}/data` accepts a `keys` map of column -> value conditions. We pass the primary key column(s) for each row. Rows are deleted one at a time.

In [ ]:
def delete_all_rows(table_id: str, table_name: str, pk_cols: list[str]) -> None:
    """Delete all rows from a DBRepo table by fetching current PKs and
    deleting them one by one via DELETE /table/{tableId}/data.
    pk_cols: list of primary key column names as stored in DBRepo.
    """
    # First fetch all existing rows to get their PK values
    page, page_size = 0, 500
    all_rows = []
    while True:
        res = requests.get(
            f"{BASE_URL}/table/{table_id}/data",
            auth=AUTH, headers=HEADERS,
            params={"size": page_size, "page": page}
        )
        if not res.ok:
            raise RuntimeError(f"Could not fetch rows for '{table_name}' ({res.status_code}): {res.text[:200]}")
        data = res.json()
        content = data.get("content", data) if isinstance(data, dict) else data
        if not content:
            break
        all_rows.extend(content)
        if len(content) < page_size:
            break
        page += 1

    if not all_rows:
        print(f"  '{table_name}': no rows to delete.")
        return

    print(f"  Deleting {len(all_rows)} rows from '{table_name}'...")
    url = f"{BASE_URL}/table/{table_id}/data"
    for idx, row in enumerate(all_rows):
        keys = {col: row[col] for col in pk_cols}
        res = requests.delete(url, auth=AUTH, headers=HEADERS, json={"keys": keys})
        if not res.ok:
            raise RuntimeError(
                f"[{table_name}] delete row {idx+1}/{len(all_rows)} failed "
                f"({res.status_code}): {res.text[:300]}"
            )
        if (idx + 1) % 100 == 0 or (idx + 1) == len(all_rows):
            print(f"    {idx+1}/{len(all_rows)} deleted", end="\r")
        time.sleep(0.05)
    print(f"  ✓ '{table_name}' cleared.          ")
.
print("=== Clearing existing data from DBRepo ===")
delete_all_rows(UNEMPLOYMENT_TABLE_ID, "unemployment", ["measurement_id", "gender"])
delete_all_rows(TOURISM_TABLE_ID, "tourism", ["measurement_id"])
delete_all_rows(MEASUREMENT_INFO_TABLE_ID, "measurement_info", ["district_id", "reference_date"])
delete_all_rows(DISTRICT_TABLE_ID, "district", ["district_id"])
print("\n✓ All tables cleared. Ready to re-insert.")

=== Clearing existing data from DBRepo ===
  Deleting 24 rows from 'district'...
  ✓ 'district' cleared.          

✓ All tables cleared. Ready to re-insert.


## 6  · Insert data into DBRepo

In [12]:
def df_to_records(df: pd.DataFrame) -> list[dict]:
    """Serialise a DataFrame to JSON-safe list of dicts.
    Converts numpy scalars (int64, float64) and date objects to native Python types.
    """
    records = []
    for row in df.itertuples(index=False):
        rec = {}
        for col, val in zip(df.columns, row):
            if isinstance(val, float) and math.isnan(val):
                rec[col] = None
            elif hasattr(val, 'isoformat'):        # date / datetime
                rec[col] = val.isoformat()
            elif isinstance(val, (np.integer,)):   # np.int64, np.int32, ...
                rec[col] = int(val)
            elif isinstance(val, (np.floating,)):  # np.float64, ...
                rec[col] = float(val)
            elif isinstance(val, (np.bool_,)):
                rec[col] = bool(val)
            else:
                rec[col] = val
        records.append(rec)
    return records

In [13]:
print("=== Inserting data into DBRepo ===")

insert_rows(DISTRICT_TABLE_ID, df_to_records(district_df))
insert_rows(MEASUREMENT_INFO_TABLE_ID, df_to_records(measurement_df))
insert_rows(TOURISM_TABLE_ID, df_to_records(tourism_df))
insert_rows(UNEMPLOYMENT_TABLE_ID, df_to_records(unemployment_df))

print("\n✓ All tables inserted successfully.")

=== Inserting data into DBRepo ===
  Inserting 24 rows into '886fba43-0570-4dda-8233-4f73473ca0a4'...
  ✓ '886fba43-0570-4dda-8233-4f73473ca0a4' done.
  Inserting 552 rows into '5d6d28a7-f211-4bc4-9ada-94450cd0fe6f'...
  ✓ '5d6d28a7-f211-4bc4-9ada-94450cd0fe6f' done.
  Inserting 552 rows into 'fa5fe819-9b20-422a-a6fe-299ba0043d87'...
  ✓ 'fa5fe819-9b20-422a-a6fe-299ba0043d87' done.
  Inserting 1584 rows into '5a3f74b9-8a62-4f33-9486-6c6cd3d34cd2'...
  ✓ '5a3f74b9-8a62-4f33-9486-6c6cd3d34cd2' done.

✓ All tables inserted successfully.


## 7  · Verify views defined in views.sql

For each view we check:
- The endpoint responds with HTTP 200
- The returned columns match expectation
- Row counts and spot values are plausible
- COVID years 2020/2021 are absent where required
- District 90000 is absent where required

In [14]:
ML_FEATURE_TABLE_VIEW_ID = "2a10edf4-6645-4f6f-9bb5-83919cff02ea"
TRAIN_SPLIT_VIEW_ID = "3edb9ad6-78c1-48b7-afc6-f6c88b4a1a61"
VALIDATION_SPLIT_VIEW_ID = "abba7dcd-e5ab-4f5c-9cb8-b1fc1bb7d6b5"
TEST_SPLIT_VIEW_ID = "9469de00-ad14-489b-a8f1-73e1ec21218c"
INNER_CITY_DISTRICTS_VIEW_ID = "5a624d3a-876e-4078-83b4-c0bbec256179"
OUTER_CITY_DISTRICTS_VIEW_ID = "96116d38-a9c9-4dda-b608-6fe65ade65ce"
GENDER_DISAGGREGATED_FEATURES_VIEW_ID = "fe047ba9-61e1-433c-9bd7-52551ca15b33"

In [15]:
def get_all_view_rows(view_id: str, page_size: int = 500) -> pd.DataFrame:
    """Paginate through all rows of a view and return a single DataFrame."""
    url = f"{BASE_URL}/view/{view_id}/data"
    frames = []
    page = 0

    while True:
        res = requests.get(
            url, headers=HEADERS, auth=AUTH,
            params={"size": page_size, "page": page}
        )
        if not res.ok:
            raise RuntimeError(
                f"View '{view_id}' page {page} failed ({res.status_code}): {res.text[:300]}"
            )
        payload = res.json()
        data = payload["content"] if isinstance(payload, dict) and "content" in payload else payload
        if not data:
            break
        frames.append(pd.DataFrame(data))
        if len(data) < page_size:
            break
        page += 1
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

### 7.1 · VIEW: `ml_feature_table`

In [16]:
df = get_all_view_rows(ML_FEATURE_TABLE_VIEW_ID)

expected_cols = {"district_code", "population_avg", "reference_date", "uep_value", "uep_density", "tou_value", "tou_density"}
assert not df.empty,                                    f"{ML_FEATURE_TABLE_VIEW_ID}: no rows returned"
assert expected_cols.issubset(df.columns),              f"{ML_FEATURE_TABLE_VIEW_ID}: missing columns {expected_cols - set(df.columns)}"
assert 90000 not in df["district_code"].values,         f"{ML_FEATURE_TABLE_VIEW_ID}: district 90000 should be excluded"

df["ref_year"] = pd.to_datetime(df["reference_date"]).dt.year
assert not df["ref_year"].isin([2020, 2021]).any(),     f"{ML_FEATURE_TABLE_VIEW_ID}: COVID years 2020/2021 should be excluded"

# Expected row count: 23 districts × years in {2002..2019, 2022, 2023} = 23 × 20 = 460
n_districts = df["district_code"].nunique()
n_years = df["ref_year"].nunique()
print(f"✓ {ML_FEATURE_TABLE_VIEW_ID}")
print(f"  Rows      : {len(df)}  (expected ~460)")
print(f"  Districts : {n_districts}  (expected 23, excl. 90000)")
print(f"  Years     : {n_years}  (expected 20 - excl. 2020/2021)")
print(f"  Year range: {df['ref_year'].min()} - {df['ref_year'].max()}")
df.head(3)

✓ 2a10edf4-6645-4f6f-9bb5-83919cff02ea
  Rows      : 460  (expected ~460)
  Districts : 23  (expected 23, excl. 90000)
  Years     : 20  (expected 20 - excl. 2020/2021)
  Year range: 2002 - 2023


,district_code,population_avg,reference_date,tou_density,tou_value,uep_density,uep_value,ref_year
0,91500,79028,2017-12-31,12022.18,950086,147.13,8538,2017
1,91400,81042,2004-12-31,3268.52,264888,66.97,3763,2004
2,90100,16329,2018-12-31,182274.94,2976413,38.64,406,2018


### 7.2 · VIEW: `train_split`

In [17]:
df = get_all_view_rows(TRAIN_SPLIT_VIEW_ID)

assert not df.empty,                              f"{TRAIN_SPLIT_VIEW_ID}: no rows returned"
assert 90000 not in df["district_code"].values,   f"{TRAIN_SPLIT_VIEW_ID}: district 90000 leaked in"

df["ref_year"] = pd.to_datetime(df["reference_date"]).dt.year
assert (df["ref_year"] <= 2015).all(),            f"{TRAIN_SPLIT_VIEW_ID}: contains years > 2015"

# Expected: 23 districts × 14 years (2002-2015) = 322
print(f"✓ {TRAIN_SPLIT_VIEW_ID}")
print(f"  Rows     : {len(df)}  (expected 322)")
print(f"  Year range: {df['ref_year'].min()} - {df['ref_year'].max()}  (expected 2002-2015)")
df.head(3)

✓ 3edb9ad6-78c1-48b7-afc6-f6c88b4a1a61
  Rows     : 322  (expected 322)
  Year range: 2002 - 2015  (expected 2002-2015)


,district_code,population_avg,reference_date,tou_density,tou_value,uep_density,uep_value,ref_year
0,90900,39246,2007-12-31,12302.37,482822,41.64,1203,2007
1,92300,97442,2015-12-31,566.67,55218,78.29,4880,2015
2,91300,51306,2006-12-31,5554.20,284964,43.33,1401,2006


### 7.3 · VIEW: `validation_split`

In [18]:
df = get_all_view_rows(VALIDATION_SPLIT_VIEW_ID)

assert not df.empty,                              f"{VALIDATION_SPLIT_VIEW_ID}: no rows returned"

df["ref_year"] = pd.to_datetime(df["reference_date"]).dt.year
assert df["ref_year"].between(2016, 2018).all(),  f"{VALIDATION_SPLIT_VIEW_ID}: contains years outside 2016-2018"

# Expected: 23 × 3 = 69
print(f"✓ {VALIDATION_SPLIT_VIEW_ID}")
print(f"  Rows      : {len(df)}  (expected 69)")
print(f"  Year range: {df['ref_year'].min()} - {df['ref_year'].max()}  (expected 2016-2018)")
df.head(3)

✓ abba7dcd-e5ab-4f5c-9cb8-b1fc1bb7d6b5
  Rows      : 69  (expected 69)
  Year range: 2016 - 2018  (expected 2016-2018)


,district_code,population_avg,reference_date,tou_density,tou_value,uep_density,uep_value,ref_year
0,91500,79028,2017-12-31,12022.18,950086,147.13,8538,2017
1,92200,188828,2018-12-31,3489.96,659004,93.10,11954,2018
2,90200,105143,2017-12-31,17818.46,1873493,121.13,8973,2017


### 7.4 · VIEW: `test_split`

In [20]:
df = get_all_view_rows(TEST_SPLIT_VIEW_ID)

assert not df.empty,                                  f"{TEST_SPLIT_VIEW_ID}: no rows returned"

df["ref_year"] = pd.to_datetime(df["reference_date"]).dt.year
assert (df["ref_year"] >= 2019).all(),                f"{TEST_SPLIT_VIEW_ID}: contains years < 2019"
assert not df["ref_year"].isin([2020, 2021]).any(),   f"{TEST_SPLIT_VIEW_ID}: COVID years should be excluded"

# Expected: 23 × 3 years (2019, 2022, 2023) = 69
print(f"✓ {TEST_SPLIT_VIEW_ID}")
print(f"  Rows      : {len(df)}  (expected 69)")
print(f"  Year range: {df['ref_year'].min()} - {df['ref_year'].max()}")
print(f"  Years present: {sorted(df['ref_year'].unique())}  (expected [2019, 2022, 2023])")
df.head(3)

✓ 9469de00-ad14-489b-a8f1-73e1ec21218c
  Rows      : 69  (expected 69)
  Year range: 2019 - 2023
  Years present: [np.int32(2019), np.int32(2022), np.int32(2023)]  (expected [2019, 2022, 2023])


,district_code,population_avg,reference_date,tou_density,tou_value,uep_density,uep_value,ref_year
0,90900,41948,2022-12-31,10909.49,457626,56.64,1738,2022
1,91800,51438,2023-12-31,1326.26,68220,66.28,2343,2023
2,90400,33386,2023-12-31,17422.81,581678,78.41,1855,2023


### 7.5 · VIEW: `inner_city_districts`

In [21]:
df = get_all_view_rows(INNER_CITY_DISTRICTS_VIEW_ID)

assert not df.empty,                                  f"{INNER_CITY_DISTRICTS_VIEW_ID}: no rows returned"
assert (df["district_code"] <= 90900).all(),           f"{INNER_CITY_DISTRICTS_VIEW_ID}: contains districts > 90900"
assert 90000 not in df["district_code"].values,       f"{INNER_CITY_DISTRICTS_VIEW_ID}: district 90000 leaked in"
df["ref_year"] = pd.to_datetime(df["reference_date"]).dt.year

# Expected row count: 9 districts × years in {2002..2019, 2022, 2023} = 9 × 20 = 180
print(f"✓ {INNER_CITY_DISTRICTS_VIEW_ID}")
print(f"  Rows      : {len(df)}  (expected 180)")
print(f"  Year range: {df['ref_year'].min()} - {df['ref_year'].max()}")
df.head(3)

✓ 5a624d3a-876e-4078-83b4-c0bbec256179
  Rows      : 180  (expected 180)
  Year range: 2002 - 2023


,district_code,population_avg,reference_date,tou_density,tou_value,uep_density,uep_value,ref_year
0,90600,31245,2015-12-31,21729.78,678936,90.57,2113,2015
1,90200,105143,2017-12-31,17818.46,1873493,121.13,8973,2017
2,90600,31615,2016-12-31,21680.31,685431,89.10,2107,2016


### 7.6 · VIEW: `outer_city_districts`

In [22]:
df = get_all_view_rows(OUTER_CITY_DISTRICTS_VIEW_ID)

assert not df.empty,                                   f"{OUTER_CITY_DISTRICTS_VIEW_ID}: no rows returned"
assert (df["district_code"] >= 91000).all(),           f"{OUTER_CITY_DISTRICTS_VIEW_ID}: contains districts < 91000"
df["ref_year"] = pd.to_datetime(df["reference_date"]).dt.year

# Expected row count: 14 districts × years in {2002..2019, 2022, 2023} = 14 × 20 = 280
print(f"✓ {OUTER_CITY_DISTRICTS_VIEW_ID}")
print(f"  Rows      : {len(df)}  (expected 280)")
print(f"  Year range: {df['ref_year'].min()} - {df['ref_year'].max()}")
df.head(3)

✓ 96116d38-a9c9-4dda-b608-6fe65ade65ce
  Rows      : 280  (expected 280)
  Year range: 2002 - 2023


,district_code,population_avg,reference_date,tou_density,tou_value,uep_density,uep_value,ref_year
0,91500,79028,2017-12-31,12022.18,950086,147.13,8538,2017
1,92200,188828,2018-12-31,3489.96,659004,93.10,11954,2018
2,92300,97442,2015-12-31,566.67,55218,78.29,4880,2015


### 7.7 · VIEW: `gender_disaggregated_features`

In [23]:
df = get_all_view_rows(GENDER_DISAGGREGATED_FEATURES_VIEW_ID)

expected_cols = {"district_code", "reference_date", "gender", "uep_value", "uep_density", "tou_value", "tou_density"}
assert not df.empty,                                       f"{GENDER_DISAGGREGATED_FEATURES_VIEW_ID}: no rows returned"
assert expected_cols.issubset(df.columns),                 f"{GENDER_DISAGGREGATED_FEATURES_VIEW_ID}: missing columns {expected_cols - set(df.columns)}"
assert 90000 not in df["district_code"].values,            f"{GENDER_DISAGGREGATED_FEATURES_VIEW_ID}: district 90000 should be excluded"

df["ref_year"] = pd.to_datetime(df["reference_date"]).dt.year
assert not df["ref_year"].isin([2020, 2021]).any(),        f"{GENDER_DISAGGREGATED_FEATURES_VIEW_ID}: COVID years should be excluded"

df["gender"] = df["gender"].str.strip()
assert set(df["gender"].unique()).issubset({"Male","Female"}), \
    f"{GENDER_DISAGGREGATED_FEATURES_VIEW_ID}: 'Both' gender should be excluded; found {df['gender'].unique()}"

# Each (district, year) should appear exactly twice (Male + Female)
counts = df.groupby(["district_code", "ref_year"])["gender"].count()
assert (counts == 2).all(),  f"{GENDER_DISAGGREGATED_FEATURES_VIEW_ID}: expected 2 gender rows per (district, year)"

# Expected: 23 × 20 non-COVID years × 2 genders = 920
print(f"✓ {GENDER_DISAGGREGATED_FEATURES_VIEW_ID}")
print(f"  Rows      : {len(df)}  (expected 920)")
print(f"  Genders   : {sorted(df['gender'].unique())}  (expected ['Female', 'Male'])")
print(f"  Year range: {df['ref_year'].min()} - {df['ref_year'].max()}")
df.head(4)

✓ fe047ba9-61e1-433c-9bd7-52551ca15b33
  Rows      : 920  (expected 920)
  Genders   : ['Female', 'Male']  (expected ['Female', 'Male'])
  Year range: 2002 - 2023


,district_code,gender,population_avg,reference_date,tou_density,tou_value,uep_density,uep_value,ref_year
0,90400,Female,33394,2022-12-31,13826.55,461717,58.72,698,2022
1,90400,Male,30823,2012-12-31,19611.70,604494,58.37,619,2012
2,90800,Female,23276,2010-12-31,14714.70,342492,33.37,288,2010
3,91200,Female,87303,2009-12-31,2428.19,211988,61.73,1861,2009


## 8  · Summary

In [24]:
views_checked = [
    "ml_feature_table",
    "train_split",
    "validation_split",
    "test_split",
    "inner_city_districts",
    "outer_city_districts",
    "gender_disaggregated_features",
]

print("=" * 52)
print("DBRepo Load & View Verification - Summary")
print("=" * 52)
print(f"{'Table':<20} {'Rows':>6}")
print("-" * 28)
for name, df_ in [
    ("district", district_df),
    ("measurement_info",measurement_df),
    ("tourism", tourism_df),
    ("unemployment", unemployment_df),
]:
    print(f"  {name:<18} {len(df_):>6}")
print()
print(f"{'View':<30} {'Status':>8}")
print("-" * 40)
for v in views_checked:
    print(f"  {v:<28} {'✓ OK':>8}")
print("=" * 52)
print("All checks passed.")

DBRepo Load & View Verification - Summary
Table                  Rows
----------------------------
  district               24
  measurement_info      552
  tourism               552
  unemployment         1584

View                             Status
----------------------------------------
  ml_feature_table                 ✓ OK
  train_split                      ✓ OK
  validation_split                 ✓ OK
  test_split                       ✓ OK
  inner_city_districts             ✓ OK
  outer_city_districts             ✓ OK
  gender_disaggregated_features     ✓ OK
All checks passed.
